# FlumenData QuickStart

Welcome to FlumenData! This notebook will guide you through the basics of using the lakehouse platform.

## What you'll learn:
1. Connect to the Spark cluster
2. Explore databases and tables
3. Create a Delta Lake table
4. Query and analyze data
5. Use Delta Lake time travel

## 1. Connect to Spark Cluster

First, let's create a Spark session connected to the FlumenData cluster.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Create Spark session
spark = SparkSession.builder \
    .appName("FlumenData-QuickStart") \
    .master("spark://spark-master:7077") \
    .config("spark.submit.deployMode", "client") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

# Verify connection
print(f"✓ Spark version: {spark.version}")
print(f"✓ Catalog: {spark.conf.get('spark.sql.catalogImplementation')}")
print(f"✓ Master: {spark.conf.get('spark.master')}")

## 2. Explore Databases

FlumenData uses a 2-level namespace (database.table). Let's see what databases exist.

In [ ]:
# Show all databases
spark.sql("SHOW DATABASES").show()

In [ ]:
# Get database details
databases = spark.sql("SHOW DATABASES").collect()
print(f"Total databases: {len(databases)}")
for db in databases:
    print(f"  • {db.namespace}")

## 3. Create a Test Database and Table

Let's create a sample database and populate it with data.

In [ ]:
# Create database
spark.sql("CREATE DATABASE IF NOT EXISTS quickstart")
spark.sql("USE quickstart")

print("✓ Database 'quickstart' created and selected")

In [ ]:
# Create sample data
data = [
    (1, "Alice", "alice@example.com", "USA", "2024-01-15", 1250.50),
    (2, "Bob", "bob@example.com", "Canada", "2024-02-20", 890.25),
    (3, "Carol", "carol@example.com", "UK", "2024-03-10", 2100.00),
    (4, "David", "david@example.com", "USA", "2024-04-05", 1575.75),
    (5, "Eve", "eve@example.com", "Germany", "2024-05-12", 3200.00)
]

schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("country", StringType(), False),
    StructField("signup_date", StringType(), False),
    StructField("lifetime_value", DoubleType(), False)
])

df = spark.createDataFrame(data, schema)
df.show()

In [ ]:
# Write as Delta table
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("quickstart.customers")

print("✓ Delta table 'quickstart.customers' created")

## 4. Query and Analyze Data

Now let's query our Delta table using both SQL and DataFrame API.

In [ ]:
# SQL query
spark.sql("""
    SELECT country, 
           COUNT(*) as customer_count,
           ROUND(AVG(lifetime_value), 2) as avg_ltv,
           ROUND(SUM(lifetime_value), 2) as total_ltv
    FROM quickstart.customers
    GROUP BY country
    ORDER BY total_ltv DESC
""").show()

In [ ]:
# DataFrame API
customers_df = spark.table("quickstart.customers")

# Show schema
customers_df.printSchema()

# Basic statistics
customers_df.describe().show()

In [ ]:
# Filter high-value customers
high_value = customers_df.filter(col("lifetime_value") > 1500)
print(f"High-value customers (LTV > $1500): {high_value.count()}")
high_value.show()

## 5. Delta Lake Time Travel

Delta Lake tracks all changes. Let's update data and then travel back in time.

In [ ]:
# View table history
spark.sql("DESCRIBE HISTORY quickstart.customers").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

In [ ]:
# Update a record
spark.sql("""
    UPDATE quickstart.customers
    SET lifetime_value = 5000.00
    WHERE customer_id = 5
""")

print("✓ Updated Eve's lifetime value to $5000")

In [ ]:
# Show current data
print("Current data:")
spark.table("quickstart.customers").filter(col("customer_id") == 5).show()

In [ ]:
# Query previous version (version 0)
print("Data at version 0 (before update):")
spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("quickstart.customers") \
    .filter(col("customer_id") == 5) \
    .show()

In [ ]:
# View updated history
spark.sql("DESCRIBE HISTORY quickstart.customers").select(
    "version", "timestamp", "operation"
).show(truncate=False)

## 6. Visualization with pandas and matplotlib

Let's create a simple visualization of our data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")

# Convert to pandas for visualization
pdf = spark.table("quickstart.customers").toPandas()

# Create plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Customer count by country
country_counts = pdf['country'].value_counts()
ax1.bar(country_counts.index, country_counts.values)
ax1.set_title('Customers by Country')
ax1.set_xlabel('Country')
ax1.set_ylabel('Customer Count')

# Lifetime value by customer
ax2.barh(pdf['name'], pdf['lifetime_value'])
ax2.set_title('Lifetime Value by Customer')
ax2.set_xlabel('Lifetime Value ($)')
ax2.set_ylabel('Customer')

plt.tight_layout()
plt.show()

## 7. Access MinIO (S3) Directly

You can also access the underlying storage directly using boto3.

In [ ]:
import boto3
from botocore.client import Config

# Configure S3 client for MinIO
s3 = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin123',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# List objects in warehouse
print("Objects in lakehouse bucket:")
response = s3.list_objects_v2(Bucket='lakehouse', Prefix='warehouse/quickstart.db/')
for obj in response.get('Contents', [])[:10]:  # Show first 10
    print(f"  {obj['Key']} ({obj['Size']} bytes)")

## 8. Clean Up (Optional)

To clean up the test data, uncomment and run:

In [ ]:
# Uncomment to drop the table and database
# spark.sql("DROP TABLE IF EXISTS quickstart.customers")
# spark.sql("DROP DATABASE IF EXISTS quickstart")
# print("✓ Cleaned up test data")

## 9. Stop Spark Session

Always stop the Spark session when done.

In [ ]:
spark.stop()
print("✓ Spark session stopped")

## Next Steps

Congratulations! You've completed the FlumenData quickstart. Here's what you learned:

- ✓ Connect to Spark cluster
- ✓ Create databases and Delta tables
- ✓ Query data with SQL and DataFrame API
- ✓ Use Delta Lake time travel
- ✓ Visualize data with pandas and matplotlib
- ✓ Access S3/MinIO storage directly

### Explore More:
- Try creating your own databases and tables
- Experiment with more complex queries
- Load external data from CSV/Parquet/JSON
- Use Delta Lake merge operations
- Check the documentation at `/docs`